# Ch01-06: 범주형 인코딩 고급 기법


## 학습 목표

- Target Encoding의 원리와 정보 누출 방지 기법(smoothing, CV)을 이해한다
- Weight of Evidence(WoE)와 Information Value(IV)의 수학적 정의와 활용법을 습득한다
- CatBoost Encoding의 순서 의존 정보 누출 방지를 이해한다
- 고카디널리티 범주형 변수 처리 전략을 비교한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. Target Encoding (Mean Encoding)

범주 $c$에 대해 타겟 평균으로 인코딩: $\hat{y}_c = \frac{\sum_{i:x_i=c} y_i}{n_c}$

**Smoothing** (정보 누출 방지):

$$\hat{y}_c^{\text{smooth}} = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{\text{global}}}{n_c + m}$$

$m$은 smoothing factor. $n_c$가 작을수록 전역 평균에 가까워진다.

### 2. Weight of Evidence (WoE)

이진 분류에서 범주 $c$의 WoE:

$$\text{WoE}_c = \ln\frac{P(X=c|Y=1)}{P(X=c|Y=0)} = \ln\frac{n_c^+/N^+}{n_c^-/N^-}$$

**Information Value (IV)**: 변수의 전체 예측력

$$\text{IV} = \sum_c \left(\frac{n_c^+}{N^+} - \frac{n_c^-}{N^-}\right) \cdot \text{WoE}_c$$

| IV 범위 | 예측력 |
|---------|--------|
| < 0.02 | 없음 |
| 0.02-0.1 | 약함 |
| 0.1-0.3 | 보통 |
| 0.3-0.5 | 강함 |
| > 0.5 | 의심스러움 (과적합 가능) |

### 3. CatBoost Encoding (Ordered Target Statistics)

데이터를 랜덤 순열로 정렬 후, 각 관측치에 대해 **자기 자신 이전**의 동일 범주 타겟 통계만 사용:

$$\hat{y}_i^{\text{CB}} = \frac{\sum_{j<i, x_j=x_i} y_j + a \cdot p}{|\{j<i: x_j=x_i\}| + a}$$

$a$: prior weight, $p$: prior probability


## 구현 가이드

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import KFold

np.random.seed(42)
n = 1000
categories = np.random.choice(['A','B','C','D','E','F','G','H'], n)
cat_effect = {'A':0.8,'B':0.6,'C':0.4,'D':0.3,'E':0.2,'F':0.1,'G':0.05,'H':0.5}
probs = np.array([cat_effect[c] for c in categories])
y = (np.random.random(n) < probs).astype(int)
df = pd.DataFrame({'cat': categories, 'y': y})

# 1. Naive Target Encoding (정보 누출!)
te_naive = df.groupby('cat')['y'].mean()
df['te_naive'] = df['cat'].map(te_naive)

# 2. Smoothed Target Encoding
global_mean = y.mean()
m = 10
te_smooth = {}
for c in df['cat'].unique():
    nc = (df['cat']==c).sum()
    yc = df.loc[df['cat']==c, 'y'].mean()
    te_smooth[c] = (nc*yc + m*global_mean) / (nc + m)
df['te_smooth'] = df['cat'].map(te_smooth)

# 3. CV Target Encoding (정보 누출 방지)
df['te_cv'] = np.nan
kf = KFold(5, shuffle=True, random_state=42)
for tr_idx, val_idx in kf.split(df):
    means = df.iloc[tr_idx].groupby('cat')['y'].mean()
    df.loc[df.index[val_idx], 'te_cv'] = df.iloc[val_idx]['cat'].map(means)
df['te_cv'].fillna(global_mean, inplace=True)

print(df.groupby('cat')[['y','te_naive','te_smooth','te_cv']].mean().round(3))


---
## 연습 문제

### 문제 1 [★]

WoE/IV를 구현하고, 합성 데이터에서 각 범주형 변수의 IV를 계산하여 예측력을 평가하라.

In [ ]:
def compute_woe_iv(df, feature, target):
    # TODO
    pass


### 문제 2 [★★]

CatBoost Encoding을 직접 구현하고, Naive/Smoothed/CV/CatBoost 인코딩의 과적합 정도를 비교하라. 학습-검증 AUC 차이로 과적합을 측정.

In [ ]:
def catboost_encoding(df, cat_col, target_col, a=1):
    # TODO
    pass


### 문제 3 [★★]

고카디널리티(1000개 범주) 변수에서 One-Hot, Frequency, Target(CV), Hash 인코딩의 분류 성능과 메모리를 비교하라.

In [ ]:
# TODO: 고카디널리티 실험


### 문제 4 [★★★]

Target Encoding의 이론적 편향-분산 분해.
1) smoothing factor m에 따른 편향/분산 tradeoff를 시뮬레이션
2) 최적 m을 교차검증으로 선택하는 알고리즘 구현
3) 범주별 빈도 불균형 시 적응적 m 전략 설계

In [ ]:
# TODO: m에 따른 편향/분산
# TODO: 적응적 m


---
## 참고 자료

- Micci-Barreca, D. (2001). A Preprocessing Scheme for High-Cardinality Categorical Attributes.
- Prokhorenkova et al. (2018). CatBoost: Unbiased Boosting with Categorical Features.
- Siddiqi, N. (2006). Credit Risk Scorecards: WoE/IV methodology.
